In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("autos.csv", delimiter=',' ,encoding='latin-1')

#cleaning data
data = data.drop(columns='postalCode')
data = data.drop(columns='nrOfPictures')
data = data[data.yearOfRegistration <= 2016]
data = data[data.yearOfRegistration >= 1995]

data['dateCreated'] = pd.to_datetime(data['dateCreated'])
data['lastSeen'] = pd.to_datetime(data['lastSeen'])
data['listingAge'] = (data['lastSeen'] - data['dateCreated']) / pd.Timedelta(days=1)
#data['logPrice'] = log1p(data['price])
data = data.drop(columns=['dateCreated', 'lastSeen'])

data = data.dropna(subset=['model', 'gearbox', 'fuelType'])

data = data[data['price'] > 500]
data = data[data['price'] < 100000]

data = data[data['powerPS'] < 1000]
data = data[data['powerPS'] > 5]

data = data[data.offerType == 'Angebot']
data = data.drop(columns = 'offerType')
data['car_age'] = 2016 - data['yearOfRegistration']
data = data.drop(columns = ['seller', 'name', 'dateCrawled', 'yearOfRegistration', 'abtest', 'monthOfRegistration']) 

KeyError: "['postalCode'] not found in axis"

In [ ]:
#Splitting the data

from sklearn.model_selection import train_test_split

numeric_features = ['powerPS', 'kilometer', 'car_age', 'listingAge']
categorical_features = ['vehicleType', 'gearbox', 'model', 'fuelType', 'brand', 'notRepairedDamage', ]
#data[categorical_features] = data[categorical_features].fillna("unknown")

y = data['price'].copy()
X = data[categorical_features + numeric_features].copy()

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, random_state = 42, test_size = 0.2)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, random_state = 42, test_size = 0.2)





In [ ]:
#Standard Random Forest Regressor
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import ParameterGrid

ohe = OneHotEncoder(handle_unknown = "ignore", sparse_output = False, max_categories = 50)

seed = 42

def make_forest(criterion = "squared_error", max_depth = 3, min_samples_leaf = 5):
    prepare = ColumnTransformer([
        ("numeric", SimpleImputer(strategy = "mean"), numeric_features),
        ("categorical", ohe, categorical_features)
    ])
    return Pipeline([
        ("prepare", prepare),
        ("forest", RandomForestRegressor(criterion=criterion, max_depth=max_depth, min_samples_leaf=min_samples_leaf, n_jobs=-1, random_state = seed)),
         ])

#Defining the hyperparameters and running the model:

def holdout_table(parameter_choices):
    rows, models = [], []
    for i, settings in enumerate(ParameterGrid(parameter_choices), start=1):
        t0 = time.time()
        model = make_forest(**settings).fit(X_train, y_train)
        models.append(model)
        probs_train = model.predict(X_train)
        probs_val = model.predict(X_val)
        
        rows.append({
            "settings":settings,
            "training MAE": mean_absolute_error(y_train, probs_train), #which measure to use?!
            "validation MAE": mean_absolute_error(y_val, probs_val), #which measure to use?!
            "training RMSE": root_mean_squared_error(y_train, probs_train),
            "validation RMSE": root_mean_squared_error(y_val, probs_val),
            "val_r2": r2_score(y_val, probs_val),
            "avg_leaves": sum(t.get_n_leaves() for t in model.named_steps["forest"].estimators_) / len(model.named_steps["forest"].estimators_),
        })
        print(f"[{i}/9] Finished {settings} in {time.time() - t0:.1f}s")
    return pd.DataFrame(rows), models

depth_results, depth_models = holdout_table({
    "criterion" : ["squared_error"], "max_depth" : [3, 6, 9], "min_samples_leaf" : [5, 10, 20]
})
depth_labels = [str(settings["max_depth"]) for settings in depth_results["settings"]]

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(depth_results.round(3))




In [ ]:
#Standard HistGradientBoostRegressor
import time
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import ParameterGrid

ohe = OrdinalEncoder(handle_unknown="use_encoded_value",
    unknown_value=-1,
    encoded_missing_value=-2,
    max_categories=254,)
cat_indices = list(range(len(numeric_features), len(numeric_features) + len(categorical_features)))

seed = 42

def make_forest(loss = "absolute_error", learning_rate = 0.1, max_iter= 300, max_leaf_nodes = 31, min_samples_leaf = 5, l2_regularization=0.0,):
    prepare = ColumnTransformer([
        ("numeric", "passthrough", numeric_features),
        ("categorical", ohe, categorical_features),
    ])
    pipe = Pipeline([
        ("prepare", prepare),
        ("forest", HistGradientBoostingRegressor(loss=loss,
            learning_rate=learning_rate,
            max_iter=max_iter,
            max_leaf_nodes=max_leaf_nodes,
            min_samples_leaf=min_samples_leaf,
            l2_regularization=l2_regularization,
            categorical_features=cat_indices,
            early_stopping=True,      
            validation_fraction=0.1,
            n_iter_no_change=15,
            random_state=seed,)),
         ])
    return TransformedTargetRegressor(
        regressor=pipe,
        func=np.log1p,
        inverse_func=np.expm1,)
        
#Defining the hyperparameters and running the model:

def holdout_table(parameter_choices):
    rows, models = [], []
    for i, settings in enumerate(ParameterGrid(parameter_choices), start=1):
        t0 = time.time()
        model = make_forest(**settings).fit(X_train, y_train)
        models.append(model)
        probs_train = model.predict(X_train)
        probs_val = model.predict(X_val)

        predictors = model.regressor_.named_steps["forest"]._predictors
        
        rows.append({
            "settings":settings,
            "training MAE": mean_absolute_error(y_train, probs_train), #which measure to use?!
            "validation MAE": mean_absolute_error(y_val, probs_val), #which measure to use?!
            "training RMSE": root_mean_squared_error(y_train, probs_train),
            "validation RMSE": root_mean_squared_error(y_val, probs_val),
            "val_r2": r2_score(y_val, probs_val),
            "avg_nodes": sum(p[0].nodes.shape[0] for p in predictors) / len(predictors),
        })
        print(f"[{i}/9] Finished {settings} in {time.time() - t0:.1f}s")
    return pd.DataFrame(rows), models

depth_results, depth_models = holdout_table({
    "loss" : ["absolute_error"], "learning_rate": [0.05], "max_iter" : [1000], "max_leaf_nodes" : [127, 255], "min_samples_leaf" : [15, 20, 30], "l2_regularization": [0.1],
})
leaf_labels = [str(settings["max_leaf_nodes"]) for settings in depth_results["settings"]]

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(depth_results.round(3))


